# 일일 리포트 서비스 그래프

`report_service` 그래프의 독립 실행 버전입니다.
- **daily_report_node**: 일일 품질 종합 보고서 생성
- **critic_node**: 생성된 보고서 검수 (최대 1회 재시도)

흐름: `START → daily_node → critic_node → END`

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q qwen-vl-utils torchvision accelerate
!pip install -q bitsandbytes
!pip install -q langgraph langchain-core

In [ ]:
!hf auth login --token "YOUR_HF_TOKEN"

## 모델 로드

In [ ]:
import asyncio
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

hf_model = None
hf_processor = None


def load_model(model_id: str = "Qwen/Qwen3-VL-4B-Instruct") -> None:
    global hf_model, hf_processor

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    hf_model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        quantization_config=quantization_config,
    )
    hf_processor = AutoProcessor.from_pretrained(model_id)
    print(f"✅ {model_id} 로드 완료")


async def invoke_qwen_hf(
    system_msg: str,
    prompt_text: str,
    image_urls: list[str] = None,
) -> str:
    content = []

    if image_urls:
        for url in image_urls:
            content.append({"type": "image", "image": url, "max_pixels": 1003520})
    content.append({"type": "text", "text": prompt_text})

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": content},
    ]

    text = hf_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = hf_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    def generate():
        return hf_model.generate(
            **inputs, max_new_tokens=1500, temperature=0.2, do_sample=True
        )

    generated_ids = await asyncio.to_thread(generate)

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = hf_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    return output_text[0]


load_model()

## State 정의

In [ ]:
from typing import TypedDict, List, Optional


class DefectCount(TypedDict):
    defectType: str
    count: int


class PurchaseCount(TypedDict):
    purchase: str
    count: int


class SummaryMetrics(TypedDict):
    totalCount: int
    passCount: int
    rejectCount: int
    failedCount: int
    prevTotalCount: int
    prevRejectCount: int
    defects: List[DefectCount]
    purchases: List[PurchaseCount]


class DailyData(TypedDict):
    reportDate: str
    summaryData: SummaryMetrics


class ReportState(TypedDict):
    daily_data: Optional[DailyData]
    generated_report: str
    title: str
    retry_count: int
    critic_verdict: Optional[str]        # "PASS" | "FAIL"
    critic_issues: Optional[List[dict]]  # [{criterion, description}, ...]

## 노드 정의

### daily_report_node

In [ ]:
import json


async def daily_report_node(state: ReportState) -> dict:
    data = state.get("daily_data") or {}
    critic_issues = state.get("critic_issues") or []

    system_msg = "당신은 공장 관리자를 위한 배터리 수율 및 결함 종합 통계 분석가입니다."

    summary = data.get("summaryData", {})
    defects_json_str = json.dumps(summary.get("defects", []), ensure_ascii=False, indent=2)
    purchases_json_str = json.dumps(summary.get("purchases", []), ensure_ascii=False, indent=2)
    report_date = data.get("reportDate", "알 수 없음")

    user_msg = f"""
    [일일 통계 데이터]
    - 리포트 기준일: {report_date}
    - 총 검사 수 (totalCount): {summary.get("totalCount", 0)}
    - 양품 판정 수 (passCount): {summary.get("passCount", 0)}
    - 불량 판정 수 (rejectCount): {summary.get("rejectCount", 0)}
    - 분석 실패 수 (failedCount): {summary.get("failedCount", 0)}

    [전일 대비 비교 데이터]
    - 전일 총 검사 수 (prevTotalCount): {summary.get("prevTotalCount", 0)}
    - 전일 불량 판정 수 (prevRejectCount): {summary.get("prevRejectCount", 0)}

    [결함 유형별 발생 건수 (상세)]
    {defects_json_str}

    [제조사별 결함 발생 건수 (상세)]
    {purchases_json_str}

    위 데이터를 바탕으로 공장장 및 생산 관리자가 하루의 공정 상태를 파악하고 내일의 생산 전략을 세울 수 있는 '일일 품질 종합 보고서'를 작성해줘.
    단, 아래 주의 사항을 지켜줘.

    [주의 사항]
    ** 웹 렌더링을 위해 전체 출력 형태는 반드시 위 마크다운 템플릿 형식을 엄격하게 지켜줘. **
    ** 증감률이나 수율, 비율 등을 계산할 때는 소수점 첫째 자리까지만 간단히 표기해. **
    ** 한국어로 작성해 줘. **
    ** 없는 정보를 억지로 생성해서 추론하지 말아줘**

    [보고서 출력 마크다운 템플릿]
    반드시 아래 제공된 마크다운 및 HTML 태그 템플릿 구조를 그대로 사용하여 작성할 것. 내용만 상황에 맞게 분석하여 채워줘.

    ### {report_date}의 생산 및 수율 요약
    | 항목 | 금일 실적 | 전일 대비 분석 |
    | :--- | :--- | :--- |
    | **총 검사 수** | {summary.get("totalCount", 0)} 건 | [전일 대비 검사량 증감 요약] |
    | **양품 (PASS)** | {summary.get("passCount", 0)} 건 | - |
    | **불량 (REJECT)** | {summary.get("rejectCount", 0)} 건 | [전일 대비 불량 건수 증감 요약] |
    | **분석 실패** | {summary.get("failedCount", 0)} 건 | - |
    | **최종 수율(%)** | [금일 수율 계산]% | [전일 대비 수율 상승/하락 평가] |

    ### 주요 결함 발생 현황
    * **1위:** [가장 많이 발생한 결함명] - [건수]건 ([전체 불량 중 차지하는 비율]% 추정)
    * **2위:** [두 번째로 많이 발생한 결함명] - [건수]건 ([전체 불량 중 차지하는 비율]% 추정)
    * ...(발생한 모든 결함에 대해 위 양식으로 반복)
    * **분석 코멘트:** [오늘 발생한 결함들의 주된 특징이나 편중에 대한 1~2줄 요약]

    ### 제조사별 결함 발생 현황
    * **1위:** [가장 많이 결함이 발생한 제조사] - [건수]건 ([전체 제조사 중 차지하는 비율]% 추정)
    * **2위:** [두 번째로 많이 결함이 발생한 제조사] - [건수]건 ([전체 제조사 중 차지하는 비율]% 추정)
    * ...(결함이 발생한 모든 제조사에 대해 위 양식으로 반복)
    * **분석 코멘트:** [제조사별 불량 발생의 특징, 특정 제조사 집중 여부 등에 대한 1~2줄 요약]

    ### 총평 및 개선 제안 (Actionable Insights)
    > **일일 품질 총평:** [오늘 전체적인 품질 수준이 양호한지, 위험한지 종합 평가]
    >
    > **내일 공정 조치 권장 사항:**
    > - [ ] [최다 발생 결함을 줄이기 위해 내일 아침 점검해야 할 설비/공정 제안 1]
    > - [ ] [전일 대비 악화된 지표에 대한 개선 가이드 2]
    """

    if critic_issues:
        issues_text = "\n".join(
            f"  - 기준 {i['criterion']}: {i['description']}" for i in critic_issues
        )
        user_msg += f"""
    [이전 보고서 검수 결과 - 아래 오류를 반드시 수정하여 재작성하세요]
{issues_text}
    """

    response_text = await invoke_qwen_hf(
        system_msg=system_msg,
        prompt_text=user_msg,
    )

    return {
        "title": f"{report_date} 총 요약 보고서",
        "generated_report": response_text,
    }

### critic_node

In [ ]:
async def critic_node(state: ReportState) -> dict:
    data = state.get("daily_data") or {}
    generated_report = state.get("generated_report", "")
    retry_count = state.get("retry_count", 0)

    system_msg = "당신은 배터리 공정 일일 보고서의 품질을 검수하는 심사관입니다. 반드시 JSON 형식으로만 응답하세요."

    summary = data.get("summaryData", {})
    report_date = data.get("reportDate", "알 수 없음")
    input_data_json = json.dumps(data, ensure_ascii=False, indent=2)

    user_msg = f"""
아래 원본 데이터와 생성된 보고서를 비교하여 오류를 찾아내세요.

[원본 데이터]
{input_data_json}

[생성된 보고서]
{generated_report}

[검수 기준]
1. 섹션 완전성: 아래 4개 섹션 헤더가 모두 존재하는가?
   - ### {report_date}의 생산 및 수율 요약
   - ### 주요 결함 발생 현황
   - ### 제조사별 결함 발생 현황
   - ### 총평 및 개선 제안
2. 수치 일치: 보고서 표의 totalCount·passCount·rejectCount·failedCount가 원본과 동일한가?
   - 원본 totalCount: {summary.get("totalCount", 0)}
   - 원본 passCount: {summary.get("passCount", 0)}
   - 원본 rejectCount: {summary.get("rejectCount", 0)}
   - 원본 failedCount: {summary.get("failedCount", 0)}
3. 수율 계산: 최종 수율(%) = round(passCount / totalCount * 100, 1) 과 일치하는가?
4. 결함 순위: defects를 count 내림차순 정렬한 순서와 보고서 순위가 일치하는가?
5. 제조사 순위: purchases를 count 내림차순 정렬한 순서와 보고서 순위가 일치하는가?
6. 날조 금지: 원본 데이터에 없는 수치, 결함명, 제조사명이 보고서에 등장하지 않는가?

[출력 형식 - 반드시 JSON으로만 응답]
{{
  "verdict": "PASS" 또는 "FAIL",
  "issues": [
    {{
      "criterion": 위반 기준 번호,
      "description": "구체적으로 어떤 값이 잘못되었는지 (원본값 vs 보고서값)"
    }}
  ]
}}
verdict가 PASS이면 issues는 빈 배열.
"""

    response_text = await invoke_qwen_hf(
        system_msg=system_msg,
        prompt_text=user_msg,
    )

    try:
        text = response_text.strip()
        start = text.find("{")
        end = text.rfind("}") + 1
        result = json.loads(text[start:end])
        verdict = result.get("verdict", "PASS")
        issues = result.get("issues", [])
    except (json.JSONDecodeError, ValueError):
        # 파싱 실패 시 PASS 처리하여 무한 루프 방지
        verdict = "PASS"
        issues = []

    update = {"critic_verdict": verdict, "critic_issues": issues}

    if verdict == "FAIL":
        update["retry_count"] = retry_count + 1

    return update

## 엣지 정의

In [ ]:
from typing import Literal


def route_after_critic(state: ReportState) -> Literal["daily_node", "END"]:
    if state.get("critic_verdict") == "PASS":
        return "END"
    # FAIL: retry_count는 critic_node에서 이미 +1됨. 1회 재시도 허용
    if state.get("retry_count", 0) <= 1:
        return "daily_node"
    return "END"

## 그래프 조립

In [ ]:
from langgraph.graph import END, START, StateGraph


def build_graph():
    workflow = StateGraph(ReportState)

    workflow.add_node("daily_node", daily_report_node)
    workflow.add_node("critic_node", critic_node)

    workflow.add_edge(START, "daily_node")
    workflow.add_edge("daily_node", "critic_node")
    workflow.add_conditional_edges(
        "critic_node",
        route_after_critic,
        {"daily_node": "daily_node", "END": END},
    )

    return workflow.compile()


report_llm_model = build_graph()
report_llm_model

## 테스트

In [ ]:
mock_daily_state = {
    "daily_data": {
        "reportDate": "2026-07-27",
        "summaryData": {
            "totalCount": 15000,
            "passCount": 14750,
            "rejectCount": 240,
            "failedCount": 10,
            "prevTotalCount": 14500,
            "prevRejectCount": 150,
            "defects": [
                {"defectType": "CRACK", "count": 110},
                {"defectType": "SCRATCH", "count": 80},
                {"defectType": "DENT", "count": 50}
            ],
            "purchases": [
                {"purchase": "LG", "count": 20},
                {"purchase": "삼성", "count": 3}
            ]
        }
    },
    "generated_report": "",
    "title": "",
    "retry_count": 0,
    "critic_verdict": None,
    "critic_issues": None,
}

In [ ]:
from IPython.display import display, Markdown

print("🚀 일일 리포트 생성 중...")
result = await report_llm_model.ainvoke(mock_daily_state)

print(f"\n📋 제목: {result['title']}")
print(f"🔍 크리틱 판정: {result['critic_verdict']} (재시도 횟수: {result['retry_count']})")
if result.get('critic_issues'):
    print(f"⚠️  검수 이슈: {result['critic_issues']}")

print("\n--- 생성된 보고서 ---")
display(Markdown(result["generated_report"]))